NOTEBOOK 01: DATA ANALYSIS & PREPROCESSING

Kaggle Notebook — Retail Shelf Detection

This notebook performs:
  1. Dataset structure validation
  2. Class distribution analysis across all splits
  3. Class imbalance detection and statistics
  4. Dataset rebalancing (rescue + oversample + augment)
  5. Before/after comparison visualization

Dataset: Upload as Kaggle Dataset → /kaggle/input/retail-shelf-dataset


## CELL 1: SETUP & DATASET VALIDATION


In [ ]:
!pip install ultralytics>=8.3.0 albumentations -q

import yaml
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from collections import Counter

random.seed(42)
np.random.seed(42)

DATASET_INPUT_DIR = Path("/kaggle/input/retail-shelf-dataset")
WORKING_DIR = Path("/kaggle/working")

# Load class names from data.yaml
with open(DATASET_INPUT_DIR / "data.yaml", 'r') as f:
    cfg = yaml.safe_load(f)

CLASS_NAMES = cfg['names']
NUM_CLASSES = cfg['nc']

# Validate dataset structure
print("=" * 65)
print("  DATASET VALIDATION")
print("=" * 65)

for split in ['train', 'valid', 'test']:
    img_dir = DATASET_INPUT_DIR / split / "images"
    lbl_dir = DATASET_INPUT_DIR / split / "labels"
    imgs = list(img_dir.glob("*.jpg")) if img_dir.exists() else []
    lbls = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []
    status = "✅" if len(imgs) > 0 and abs(len(imgs) - len(lbls)) < 5 else "❌"
    print(f"  {status} {split}: {len(imgs)} images, {len(lbls)} labels")

print(f"\n  Classes: {NUM_CLASSES}")
print(f"  Names: {CLASS_NAMES[:5]}... (showing first 5)")
print("=" * 65)

## CELL 2: CLASS DISTRIBUTION ANALYSIS


In [ ]:
def count_labels(label_dir):
    """Count instances of each class in a label directory."""
    counts = Counter()
    total_objects = 0
    total_images = 0
    empty_images = 0

    label_dir = Path(label_dir)
    if not label_dir.exists():
        return counts, total_objects, total_images, empty_images

    for label_file in sorted(label_dir.glob("*.txt")):
        total_images += 1
        file_has_objects = False
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    counts[cls_id] += 1
                    total_objects += 1
                    file_has_objects = True
        if not file_has_objects:
            empty_images += 1

    return counts, total_objects, total_images, empty_images


# Count for each split
splits = {
    'train': DATASET_INPUT_DIR / "train" / "labels",
    'valid': DATASET_INPUT_DIR / "valid" / "labels",
    'test':  DATASET_INPUT_DIR / "test"  / "labels",
}

all_counts = {}
for split_name, label_dir in splits.items():
    counts, total_obj, total_img, empty = count_labels(label_dir)
    all_counts[split_name] = counts
    print(f"\n📁 {split_name.upper()}:")
    print(f"   Images: {total_img} | Objects: {total_obj} | Empty: {empty}")
    print(f"   Avg objects/image: {total_obj / max(total_img, 1):.1f}")

# Build full DataFrame
rows = []
for cls_id in range(NUM_CLASSES):
    cls_name = CLASS_NAMES[cls_id]
    train_count = all_counts['train'].get(cls_id, 0)
    valid_count = all_counts['valid'].get(cls_id, 0)
    test_count  = all_counts['test'].get(cls_id, 0)
    total = train_count + valid_count + test_count
    rows.append({
        'class_id': cls_id, 'class_name': cls_name,
        'train': train_count, 'valid': valid_count,
        'test': test_count, 'total': total,
    })

df = pd.DataFrame(rows).sort_values('total', ascending=False).reset_index(drop=True)
total_all = df['total'].sum()

# Print full table
print(f"\n{'='*75}")
print(f"  CLASS DISTRIBUTION — ALL SPLITS")
print(f"{'='*75}")
print(f"  {'Rank':<5} {'Class':<10} {'Train':>7} {'Valid':>7} {'Test':>7} {'Total':>7} {'Share%':>8}")
print(f"{'-'*75}")
for i, row in df.iterrows():
    pct = row['total'] / total_all * 100 if total_all > 0 else 0
    print(f"  {i+1:<5} {row['class_name']:<10} {row['train']:>7} {row['valid']:>7} "
          f"{row['test']:>7} {row['total']:>7} {pct:>7.2f}%")

# Key statistics
max_cls = df.iloc[0]
min_cls = df[df['total'] > 0].iloc[-1]
ratio = max_cls['total'] / max(min_cls['total'], 1)

print(f"\n{'='*55}")
print(f"  IMBALANCE STATISTICS")
print(f"{'='*55}")
print(f"  Most common:   {max_cls['class_name']} ({max_cls['total']} instances)")
print(f"  Least common:  {min_cls['class_name']} ({min_cls['total']} instances)")
print(f"  Imbalance ratio: {ratio:.1f}x")
print(f"  Median: {df['total'].median():.0f} | Mean: {df['total'].mean():.1f} | Std: {df['total'].std():.1f}")
for t in [5, 10, 20]:
    print(f"  Classes with < {t} TRAIN samples: {len(df[df['train'] < t])}/{NUM_CLASSES}")
print(f"{'='*55}")

df.to_csv(WORKING_DIR / "class_distribution.csv", index=False)
print(f"\n📄 Saved: class_distribution.csv")

## CELL 3: VISUALIZATIONS


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# 1. Horizontal bar — all classes
ax1 = axes[0, 0]
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(df)))[::-1]
ax1.barh(range(len(df)), df['total'].values, color=colors, edgecolor='white', linewidth=0.3)
ax1.set_yticks(range(len(df)))
ax1.set_yticklabels(df['class_name'].values, fontsize=7)
ax1.invert_yaxis()
ax1.set_xlabel("Total Instances", fontweight='bold')
ax1.set_title("Class Distribution (All Splits)", fontweight='bold')
ax1.axvline(x=df['total'].mean(), color='red', linestyle='--', alpha=0.7, label=f"Mean: {df['total'].mean():.0f}")
ax1.legend(fontsize=9)
ax1.grid(axis='x', alpha=0.3)

# 2. Split comparison
ax2 = axes[0, 1]
x = np.arange(len(df))
w = 0.3
ax2.bar(x - w, df['train'], w, label='Train', color='#3498db', alpha=0.8)
ax2.bar(x, df['valid'], w, label='Valid', color='#2ecc71', alpha=0.8)
ax2.bar(x + w, df['test'], w, label='Test', color='#e74c3c', alpha=0.8)
ax2.set_xlabel("Class (sorted by total)", fontweight='bold')
ax2.set_ylabel("Instances", fontweight='bold')
ax2.set_title("Split Distribution per Class", fontweight='bold')
ax2.set_xticks(x[::5])
ax2.set_xticklabels(df['class_name'].values[::5], rotation=45, ha='right', fontsize=7)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# 3. Box plot
ax3 = axes[1, 0]
bp = ax3.boxplot([df['train'], df['valid'], df['test'], df['total']],
                  labels=['Train', 'Valid', 'Test', 'Total'], patch_artist=True)
for patch, c in zip(bp['boxes'], ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
ax3.set_ylabel("Instances per Class", fontweight='bold')
ax3.set_title("Distribution Spread", fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# 4. Pareto
ax4 = axes[1, 1]
cumulative = np.cumsum(df['total'].values) / df['total'].sum() * 100
ax4.fill_between(range(len(cumulative)), cumulative, alpha=0.3, color='#3498db')
ax4.plot(range(len(cumulative)), cumulative, 'o-', color='#2980b9', markersize=4)
ax4.axhline(y=80, color='red', linestyle='--', alpha=0.5, label='80% of data')
ax4.set_xlabel("Number of Classes (sorted)", fontweight='bold')
ax4.set_ylabel("Cumulative %", fontweight='bold')
ax4.set_title("Cumulative Distribution (Pareto)", fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(WORKING_DIR / "class_imbalance_analysis.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 Charts saved!")

## CELL 4: DATASET REBALANCING


In [ ]:
BALANCED_DIR = WORKING_DIR / "balanced_dataset"

# Copy dataset to writable directory
if BALANCED_DIR.exists():
    shutil.rmtree(BALANCED_DIR)

print("Copying dataset to working directory...")
shutil.copytree(DATASET_INPUT_DIR, BALANCED_DIR)
print(f"✅ Copied to: {BALANCED_DIR}")

TRAIN_IMG = BALANCED_DIR / "train" / "images"
TRAIN_LBL = BALANCED_DIR / "train" / "labels"
VALID_IMG = BALANCED_DIR / "valid" / "images"
VALID_LBL = BALANCED_DIR / "valid" / "labels"

# ── Step 1: Rescue zero-train classes from valid ──
print("\n--- Step 1: Rescuing zero-train classes ---")

def count_train():
    counts = Counter()
    for lbl in TRAIN_LBL.glob("*.txt"):
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[int(parts[0])] += 1
    return counts

initial = count_train()
zero_classes = {c for c in range(NUM_CLASSES) if initial.get(c, 0) == 0}
print(f"  Zero-train classes: {[CLASS_NAMES[c] for c in zero_classes]}")

rescued = 0
for lbl_file in sorted(VALID_LBL.glob("*.txt")):
    with open(lbl_file) as f:
        file_classes = {int(line.strip().split()[0]) for line in f if line.strip()}
    if file_classes & zero_classes:
        img_name = lbl_file.stem + ".jpg"
        src_img = VALID_IMG / img_name
        if src_img.exists():
            shutil.copy2(src_img, TRAIN_IMG / f"rescued_{img_name}")
            shutil.copy2(lbl_file, TRAIN_LBL / f"rescued_{lbl_file.name}")
            rescued += 1
            print(f"  ✅ Rescued {img_name} → {[CLASS_NAMES[c] for c in file_classes & zero_classes]}")
print(f"  Total rescued: {rescued}")

# ── Step 2: Oversample minority classes ──
print("\n--- Step 2: Oversampling minority classes ---")

TARGET_MIN = 30
current = count_train()

# Build class→image index
class_to_images = {c: [] for c in range(NUM_CLASSES)}
for lbl in sorted(TRAIN_LBL.glob("*.txt")):
    with open(lbl) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_to_images[int(parts[0])].append(lbl.stem)

total_dup = 0
for cls_id in range(NUM_CLASSES):
    cur = current.get(cls_id, 0)
    if cur >= TARGET_MIN:
        continue
    images = class_to_images[cls_id]
    if not images:
        print(f"  ⚠️  {CLASS_NAMES[cls_id]}: No images, cannot oversample")
        continue
    avg_per = cur / max(len(images), 1) or 1
    n_dup = int(np.ceil((TARGET_MIN - cur) / avg_per))
    for i in range(n_dup):
        src = random.choice(images)
        src_img = TRAIN_IMG / f"{src}.jpg"
        src_lbl = TRAIN_LBL / f"{src}.txt"
        if src_img.exists():
            new = f"os_{cls_id}_{i}_{src}"
            shutil.copy2(src_img, TRAIN_IMG / f"{new}.jpg")
            shutil.copy2(src_lbl, TRAIN_LBL / f"{new}.txt")
            total_dup += 1
    print(f"  {CLASS_NAMES[cls_id]}: {cur} → ~{cur + int(n_dup * avg_per)} (+{n_dup} images)")
print(f"  Total duplicated: {total_dup}")

# ── Step 3: Targeted augmentation ──
print("\n--- Step 3: Targeted augmentation ---")

def augment_image(img_path, lbl_path, aug_idx):
    img = cv2.imread(str(img_path))
    if img is None:
        return False
    aug = random.choice(['brightness', 'contrast', 'hflip', 'color', 'blur', 'noise'])
    new_stem = f"aug_{aug_idx}_{img_path.stem}"
    new_img = img_path.parent / f"{new_stem}.jpg"
    new_lbl = lbl_path.parent / f"{new_stem}.txt"

    if aug == 'brightness':
        img = np.clip(img * random.uniform(0.6, 1.4), 0, 255).astype(np.uint8)
    elif aug == 'contrast':
        m = np.mean(img)
        img = np.clip((img - m) * random.uniform(0.7, 1.3) + m, 0, 255).astype(np.uint8)
    elif aug == 'hflip':
        img = cv2.flip(img, 1)
        lines = []
        with open(lbl_path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) >= 5:
                    p[1] = str(1.0 - float(p[1]))
                    lines.append(' '.join(p))
        with open(new_lbl, 'w') as f:
            f.write('\n'.join(lines) + '\n')
        cv2.imwrite(str(new_img), img)
        return True
    elif aug == 'color':
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:,:,0] = (hsv[:,:,0] + random.uniform(-10, 10)) % 180
        hsv[:,:,1] = np.clip(hsv[:,:,1] * random.uniform(0.7, 1.3), 0, 255)
        img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    elif aug == 'blur':
        img = cv2.GaussianBlur(img, (random.choice([3, 5]), random.choice([3, 5])), 0)
    elif aug == 'noise':
        noise = np.random.normal(0, 10, img.shape).astype(np.uint8)
        img = cv2.add(img, noise)

    cv2.imwrite(str(new_img), img)
    shutil.copy2(lbl_path, new_lbl)
    return True

# Re-index and augment
current = count_train()
class_to_images = {c: [] for c in range(NUM_CLASSES)}
for lbl in TRAIN_LBL.glob("*.txt"):
    with open(lbl) as f:
        for line in f:
            p = line.strip().split()
            if p:
                class_to_images[int(p[0])].append(lbl.stem)

total_aug = 0
for cls_id in range(NUM_CLASSES):
    cur = current.get(cls_id, 0)
    if cur >= TARGET_MIN:
        continue
    images = list(set(class_to_images[cls_id]))
    if not images:
        continue
    for i in range(min(TARGET_MIN - cur, 10)):
        src = random.choice(images)
        if augment_image(TRAIN_IMG / f"{src}.jpg", TRAIN_LBL / f"{src}.txt", f"{cls_id}_{i}"):
            total_aug += 1
print(f"  Augmented images created: {total_aug}")

# Final summary
final = count_train()
print(f"\n{'='*60}")
print(f"  REBALANCING COMPLETE")
print(f"{'='*60}")
print(f"  Training images: {len(list(TRAIN_IMG.glob('*.jpg')))} (was 924)")
print(f"  Training instances: {sum(final.values())} (was 3979)")
print(f"  Min instances: {min(final.values()) if final else 0}")
print(f"  Max instances: {max(final.values()) if final else 0}")
new_ratio = max(final.values()) / max(min(final.values()), 1) if final else 0
print(f"  Imbalance ratio: {new_ratio:.1f}x (was 221.5x)")
print(f"{'='*60}")

## CELL 5: BEFORE/AFTER COMPARISON CHART


In [ ]:
original_train = {r['class_id']: r['train'] for _, r in df.iterrows()}
new_counts = count_train()

rows_cmp = []
for cls_id in range(NUM_CLASSES):
    rows_cmp.append({
        'class': CLASS_NAMES[cls_id],
        'before': original_train.get(cls_id, 0),
        'after': new_counts.get(cls_id, 0),
    })
df_cmp = pd.DataFrame(rows_cmp)

fig, axes = plt.subplots(1, 2, figsize=(22, 14))

for ax, col, title, color_label in [
    (axes[0], 'before', 'BEFORE Rebalancing', '#e74c3c'),
    (axes[1], 'after', 'AFTER Rebalancing', '#2ecc71'),
]:
    sorted_df = df_cmp.sort_values(col, ascending=True)
    colors = ['#e74c3c' if v < 20 else '#f39c12' if v < 30 else '#2ecc71' for v in sorted_df[col]]
    ax.barh(range(len(sorted_df)), sorted_df[col], color=colors, edgecolor='white', linewidth=0.3)
    ax.set_yticks(range(len(sorted_df)))
    ax.set_yticklabels(sorted_df['class'], fontsize=7)
    ax.set_xlabel("Training Instances", fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold', color=color_label)
    ax.axvline(x=30, color='red', linestyle='--', alpha=0.5, label='Target min (30)')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle("Class Distribution: Before vs After Rebalancing", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(WORKING_DIR / "rebalancing_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Print change table
print(f"\n{'='*55}")
print(f"  {'Class':<10} {'Before':>8} {'After':>8} {'Change':>10}")
print(f"{'-'*55}")
for _, r in df_cmp.sort_values('after', ascending=False).iterrows():
    d = r['after'] - r['before']
    print(f"  {r['class']:<10} {r['before']:>8} {r['after']:>8} {'+' if d > 0 else ''}{d:>9}")
print(f"{'='*55}")
print(f"\n✅ Balanced dataset ready at: {BALANCED_DIR}")

## CELL 6: EXPORT BALANCED DATASET


In [ ]:
# Update data.yaml with correct paths
with open(BALANCED_DIR / "data.yaml", 'r') as f:
    cfg = yaml.safe_load(f)

cfg['train'] = str(BALANCED_DIR / "train" / "images")
cfg['val'] = str(BALANCED_DIR / "valid" / "images")
cfg['test'] = str(BALANCED_DIR / "test" / "images")

with open(BALANCED_DIR / "data.yaml", 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

# Zip for download or use in notebook 02
shutil.make_archive(str(WORKING_DIR / "balanced_dataset"), 'zip', str(BALANCED_DIR))

from IPython.display import FileLink
print("✅ Balanced dataset exported!")
print("📦 Download:")
display(FileLink("balanced_dataset.zip"))